# LLM Prompt Runner

Run prompts on different LLM models (Colab AI, OpenAI). Experiments and prompts are defined in `execution/experiments.py` and stored in the database; the notebook loads them and writes results back to the DB.

In [1]:
# Install dependencies
%pip install openai psycopg2-binary -q

from datetime import datetime
from integrations.colab_llm_provider import ColabLLMProvider
from integrations.openai_llm_provider import OpenAILLMProvider
from execution.output_result_generator import save_results
from execution.results_repository import (
    list_experiments,
    get_experiment,
    get_prompts,
    create_experiment,
    add_prompts_bulk,
)
from constants.constants import (
    MODEL_PROVIDERS,
    MODEL_KEYS,
    TEMPERATURE,
    TOP_P,
    MAX_TOKENS,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 30.6 MB/s eta 0:00:00a 0:00:01


ModuleNotFoundError: No module named 'integrations'

## Execution

### 0. Create tables and seed from experiments.py (run once)

Creates `experiment_setup`, `prompt`, and `llm_prompt_results` if they do not exist, then inserts experiment definitions from `execution/experiments.py`. Existing experiment names are skipped.

### 1. Available models

In [ ]:
from execution.setup_database import run_setup

run_setup()

In [ ]:
print("Available models (use MODEL_KEYS[i] in step 1):")
for i, k in enumerate(MODEL_KEYS):
    print(f"  [{i}] {k} ({MODEL_PROVIDERS[k]})")

### 1. Choose or create experiment (model + decoding from DB)

In [ ]:
experiments = list_experiments()
for exp in experiments:
    print(f"  id={exp['id']} | {exp['name']} | {exp['model_key']} | t={exp['temperature']} top_p={exp['top_p']} max_tokens={exp['max_tokens']}")
if not experiments:
    print("  No experiments yet. Create one in the next cell with create_experiment() and add_prompts_bulk().")

In [ ]:
EXPERIMENT_ID = 1  # use an existing id from list_experiments(), or create one below

# Optional: create a new experiment and add prompts
# EXPERIMENT_ID = create_experiment(
#     name="run_gemini_flash_2024",
#     model_key=MODEL_KEYS[0],
#     temperature=TEMPERATURE,
#     top_p=TOP_P,
#     max_tokens=MAX_TOKENS,
# )
# add_prompts_bulk(EXPERIMENT_ID, ["Your first prompt here.", "Second prompt."])

experiment = get_experiment(EXPERIMENT_ID)
if experiment:
    print(f"Experiment: {experiment['name']} (id={EXPERIMENT_ID})")
    print(f"Model: {experiment['model_key']} | temperature={experiment['temperature']} | top_p={experiment['top_p']} | max_tokens={experiment['max_tokens']}")
else:
    print("No experiment with that id. Create one with create_experiment() and add_prompts_bulk().")

### 2. Load prompts for this experiment (from DB)

In [ ]:
prompts_for_experiment = get_prompts(EXPERIMENT_ID)
PROMPTS = [p["prompt_text"] for p in prompts_for_experiment]
PROMPT_IDS = [p["id"] for p in prompts_for_experiment]
print(f"Loaded {len(PROMPTS)} prompt(s) for experiment {EXPERIMENT_ID}.")

### 3. Run and save

In [ ]:
def _get_llm(model_key: str):
    if model_key not in MODEL_PROVIDERS:
        raise ValueError(f"Model '{model_key}' not found. Available: {list(MODEL_PROVIDERS.keys())}")
    provider = MODEL_PROVIDERS[model_key]
    if provider == "colab":
        return ColabLLMProvider(model_key)
    if provider == "openai":
        return OpenAILLMProvider(model_key)
    raise ValueError(f"Unsupported provider: {provider}")


def _format_result(prompt, response, model_key, model_name, start_time, duration_seconds, success, prompt_id, error=None):
    return {
        "prompt": prompt,
        "response": response,
        "model_key": model_key,
        "model_name": model_name,
        "timestamp": start_time.isoformat(),
        "duration_seconds": duration_seconds,
        "success": success,
        "prompt_id": prompt_id,
        "error": error,
    }


def _execute_prompt(model_key: str, prompt: str, temperature: float, top_p: float, max_tokens: int, prompt_id: int):
    llm = _get_llm(model_key)
    start_time = datetime.now()
    try:
        response = llm.generate(prompt, temperature=temperature, top_p=top_p, max_tokens=max_tokens)
        duration = (datetime.now() - start_time).total_seconds()
        return _format_result(prompt, response, model_key, llm.model_name, start_time, duration, True, prompt_id)
    except Exception as e:
        duration = (datetime.now() - start_time).total_seconds()
        return _format_result(prompt, None, model_key, llm.model_name, start_time, duration, False, prompt_id, error=str(e))


results = []
for prompt_text, prompt_id in zip(PROMPTS, PROMPT_IDS):
    result = _execute_prompt(
        experiment["model_key"],
        prompt_text,
        experiment["temperature"],
        experiment["top_p"],
        experiment["max_tokens"],
        prompt_id,
    )
    results.append(result)

if results:
    save_results(EXPERIMENT_ID, results, experiment["model_key"], results[0]["model_name"])

## Configuration

### Adding new models

- Colab: Edit `integrations/colab_llm_provider.py` and add to `COLAB_MODELS`
- OpenAI: Edit `integrations/openai_llm_provider.py` and add to `OPENAI_MODELS`

Models are discovered automatically from the integration files.

### Experiments and prompts (inserted in the database)

Defined in `execution/experiments.py`. Each experiment has `name`, `model_key`, `temperature`, `top_p`, `max_tokens`, and prompts via:

- **`prompts`**: list of strings (each string is one prompt; multi-line allowed).
- **`prompts_file`**: filename in `prompts/` (e.g. `"test-prompt.txt"`). The file is read and split by a line containing only `---`; each block is one prompt (multi-line allowed). If there is no `---`, the whole file is one prompt.

Run the "Create tables and seed" cell (or `python -m execution.setup_database`) once to create tables and insert these experiments and prompts. Existing experiment names are skipped.

### Project structure

```
constants/          # constants and DB config
integrations/       # base_llm_provider, colab_llm_provider, openai_llm_provider
execution/
  experiments.py   # experiment definitions for DB seed
  setup_database.py
  results_repository.py
  output_result_generator.py
  runner.ipynb
prompts/           # optional .txt files referenced by experiments.py (prompts_file)
```